# A/B Test — Average Bidding vs. Maximum Bidding

## Business Problem

A social media platform introduced **Average Bidding** as an alternative to the existing **Maximum Bidding** ad delivery mechanism. A customer ran a 40-day controlled experiment to evaluate whether the new mechanism generates better outcomes.

- **Control Group**: Maximum Bidding (existing system)
- **Test Group**: Average Bidding (new system)

## Dataset Variables

| Variable | Description |
|----------|-------------|
| Impression | Number of times the ad was shown |
| Click | Number of clicks on the ad |
| Purchase | Number of purchases made |
| Earning | Revenue generated |

## Analytical Framework

For each variable:
1. **Shapiro-Wilk test** → normality assumption
2. **Levene test** → variance homogeneity (only if both groups normal)
3. **Student t-test** (equal var) or **Welch t-test** (unequal var) → hypothesis test
4. **Mann-Whitney U** as fallback if normality not satisfied

α = 0.05 throughout.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro, levene, ttest_ind, mannwhitneyu

pd.set_option('display.float_format', lambda x: '%.2f' % x)

df_control = pd.read_excel('ab_testing.xlsx', sheet_name='Control Group')
df_test    = pd.read_excel('ab_testing.xlsx', sheet_name='Test Group')

df_control = df_control[['Impression', 'Click', 'Purchase', 'Earning']]
df_test    = df_test[['Impression', 'Click', 'Purchase', 'Earning']]

print('Control Group shape:', df_control.shape)
print('Test Group shape:   ', df_test.shape)

## 1. Descriptive Statistics

In [ ]:
desc = pd.DataFrame({
    'Control (Max Bid)': df_control.mean(),
    'Test (Avg Bid)':    df_test.mean(),
})
desc['Change %'] = ((desc['Test (Avg Bid)'] - desc['Control (Max Bid)']) 
                    / desc['Control (Max Bid)'] * 100).map('{:+.2f}%'.format)

print(desc.to_string())

**Preliminary observations (before testing):**
- Average bidding shows ads to **+18.48%** more users (more impressions)
- Average bidding receives **-22.21%** fewer clicks → lower click-through rate
- Purchase count is up **+5.67%** (modest)
- Revenue is up **+31.77%** (striking)

The pattern suggests average bidding reaches a broader but more commercially relevant audience. Whether these differences are statistically significant is what we test below.

In [ ]:
variables = ['Impression', 'Click', 'Purchase', 'Earning']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, var in enumerate(variables):
    ax = axes[i]
    ax.boxplot(
        [df_control[var], df_test[var]],
        labels=['Control\n(Max Bid)', 'Test\n(Avg Bid)'],
        patch_artist=True,
        boxprops=dict(facecolor='#4C72B0', alpha=0.6),
        medianprops=dict(color='black', linewidth=2)
    )
    pct = (df_test[var].mean() - df_control[var].mean()) / df_control[var].mean() * 100
    ax.set_title(f'{var}  ({pct:+.1f}%)', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Control vs. Test Group — All Variables', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('ab_test_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Hypothesis Testing — All Variables

**Null Hypothesis (H₀):** There is no statistically significant difference between Maximum Bidding and Average Bidding.  
**Alternative Hypothesis (H₁):** There is a statistically significant difference.

Decision rule: reject H₀ if p-value < 0.05.

In [ ]:
def ab_test(control, test, var_name):
    print(f"\n{'='*60}")
    print(f"  VARIABLE: {var_name}")
    print(f"{'='*60}")
    
    pct = (test.mean() - control.mean()) / control.mean() * 100
    print(f"  Mean Control : {control.mean():.2f}")
    print(f"  Mean Test    : {test.mean():.2f}")
    print(f"  Change       : {pct:+.2f}%")

    # Step 1: Normality
    stat_c, p_c = shapiro(control)
    stat_t, p_t = shapiro(test)
    print(f"\n  [1] Shapiro-Wilk")
    print(f"      Control: stat={stat_c:.4f}, p={p_c:.4f} → {'Normal' if p_c>0.05 else 'NOT Normal'}")
    print(f"      Test   : stat={stat_t:.4f}, p={p_t:.4f} → {'Normal' if p_t>0.05 else 'NOT Normal'}")

    both_normal = (p_c > 0.05) and (p_t > 0.05)

    if both_normal:
        # Step 2: Variance
        stat_l, p_l = levene(control, test)
        equal_var = p_l > 0.05
        print(f"\n  [2] Levene")
        print(f"      stat={stat_l:.4f}, p={p_l:.4f} → {'Equal variances' if equal_var else 'Unequal variances'}")

        # Step 3: t-test
        stat_tt, p_tt = ttest_ind(control, test, equal_var=equal_var)
        method = 'Student t-test (pooled)' if equal_var else 'Welch t-test'
        print(f"\n  [3] {method}")
        print(f"      stat={stat_tt:.4f}, p={p_tt:.4f}")
        if p_tt < 0.05:
            print(f"      → H0 REJECTED: Significant difference detected.")
        else:
            print(f"      → H0 NOT REJECTED: No significant difference.")
        return p_tt, equal_var
    else:
        stat_mw, p_mw = mannwhitneyu(control, test, alternative='two-sided')
        print(f"\n  [2+3] Mann-Whitney U (non-parametric fallback)")
        print(f"        stat={stat_mw:.4f}, p={p_mw:.4f}")
        if p_mw < 0.05:
            print(f"      → H0 REJECTED: Significant difference detected.")
        else:
            print(f"      → H0 NOT REJECTED: No significant difference.")
        return p_mw, None

results = {}
for var in variables:
    p, ev = ab_test(df_control[var], df_test[var], var)
    results[var] = p

## 3. Results Summary

In [ ]:
summary_data = {
    'Control Mean': df_control.mean(),
    'Test Mean':    df_test.mean(),
    'Change %':     ((df_test.mean() - df_control.mean()) / df_control.mean() * 100),
    'p-value':      pd.Series(results),
    'Significant':  pd.Series({k: 'YES' if v < 0.05 else 'NO' for k, v in results.items()})
}

summary = pd.DataFrame(summary_data)
summary['Change %'] = summary['Change %'].map('{:+.2f}%'.format)
summary['p-value']  = summary['p-value'].map('{:.4f}'.format)
print(summary.to_string())

## 4. Interpretation & Business Recommendation

| Variable | Direction | Magnitude | Significant | Interpretation |
|----------|-----------|-----------|-------------|----------------|
| Impression | ↑ | +18.48% | YES | Average bidding reaches significantly more users |
| Click | ↓ | -22.21% | YES | Click-through rate is significantly lower |
| Purchase | ↑ | +5.67% | NO | Purchase uplift is not statistically proven |
| Earning | ↑ | +31.77% | YES | Revenue is significantly and substantially higher |

### Economic Interpretation

The data reveals a coherent story:

**Average bidding operates at a lower cost-per-impression**, exposing the ad to a broader audience. This wider audience has a lower click-through rate (fewer people click), but the users who *do* engage appear to be higher-value — the revenue uplift (+31.77%) vastly exceeds the purchase count difference (+5.67%, not significant).

This implies **higher revenue per transaction** under average bidding — the platform is optimising for value-weighted audiences, not raw click volume.

### Decision

> **Switch to Average Bidding.**

The only metric that directly matters to the business — **Earning** — shows a statistically significant, economically substantial improvement of **+31.77%**. The lower click volume is not a problem; it is a consequence of more efficient ad placement.

Purchase count does not show a statistically significant difference (p=0.35), so it should not be used to argue either direction. More data collection on the Purchase variable could be warranted if the business wants to decompose the revenue gain into *conversion rate* vs. *basket size* effects — but this should not delay the decision on bidding strategy.
